In [3]:

import sys, os, json, pickle, numpy as np
sys.path.insert(0, '.')
from utils import FeatureExtractor
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Load dataset
print('Loading dataset...')
with open('../security/embeddings/malicious_prompts_combined.json',encoding = 'utf-8') as f:
    mal = json.load(f)['prompts'][:1000]
with open('../security/embeddings/benign_prompts.json', encoding = 'utf-8') as f:
    ben = json.load(f)['prompts'][:500]

# Extract features
print('Extracting features...')
extractor = FeatureExtractor()
X = []
y = []
for p in mal + ben:
    feat = extractor.extract_all_features(p)
    X.append(extractor.features_to_array(feat))
    y.append(1 if len(y) < len(mal) else 0)

X = np.array(X)
y = np.array(y)

# Train
print('Training model...')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = xgb.XGBClassifier(max_depth=6, n_estimators=100, learning_rate=0.1, n_jobs=-1)
model.fit(X_train, y_train)

# Save
os.makedirs('models', exist_ok=True)
with open('models/xgboost_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print(f'✅ Model trained! Accuracy: {model.score(X_test, y_test):.2%}')
print('Saved to: models/xgboost_model.pkl')


Loading dataset...
Extracting features...
Training model...
✅ Model trained! Accuracy: 80.33%
Saved to: models/xgboost_model.pkl
